# Sprite Generator - GPU Training Kernel
Trains VQ-VAE + Transformer on HF Dataset. Pushes checkpoints to HF Hub.

In [ ]:
import os
import sys
import torch
import json
from pathlib import Path

# Install dependencies
!pip install -q datasets huggingface_hub torchvision pillow tqdm

# Clone repo to get source code
!git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_DATASET = os.environ.get("HF_DATASET", "darklord8777/sprites")
HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "darklord8777/sprite-generator-model")
TRAIN_VQVAE = os.environ.get("TRAIN_VQVAE", "1") == "1"
TRAIN_TRANSFORMER = os.environ.get("TRAIN_TRANSFORMER", "1") == "1"
EPOCHS_VQVAE = int(os.environ.get("EPOCHS_VQVAE", "50"))
EPOCHS_TRANSFORMER = int(os.environ.get("EPOCHS_TRANSFORMER", "50"))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Stage 1: Train VQ-VAE

In [ ]:
if TRAIN_VQVAE:
    from models.vqvae.model import VQVAE
    from models.vqvae.train import main as train_vqvae
    
    sys.argv = ["train_vqvae.py",
        "--dataset", HF_DATASET,
        "--epochs", str(EPOCHS_VQVAE),
        "--checkpoint-dir", "/kaggle/working/checkpoints/vqvae",
        "--hf-repo", HF_MODEL_REPO,
        "--hf-token", HF_TOKEN,
    ]
    train_vqvae()
    print("VQ-VAE training complete!")

## Stage 2: Train Transformer Prior

In [ ]:
if TRAIN_TRANSFORMER:
    from models.transformer.model import SpriteTransformer
    from models.transformer.train import main as train_transformer
    
    # Find latest VQ-VAE checkpoint
    vqvae_ckpts = sorted(Path("/kaggle/working/checkpoints/vqvae").glob("*.pt"))
    if not vqvae_ckpts:
        # Try to download from HF Hub
        from huggingface_hub import hf_hub_download
        vqvae_path = hf_hub_download(HF_MODEL_REPO, "vqvae_latest.pt", token=HF_TOKEN)
    else:
        vqvae_path = str(vqvae_ckpts[-1])
    
    sys.argv = ["train_transformer.py",
        "--dataset", HF_DATASET,
        "--vqvae-checkpoint", vqvae_path,
        "--epochs", str(EPOCHS_TRANSFORMER),
        "--checkpoint-dir", "/kaggle/working/checkpoints/transformer",
        "--hf-repo", HF_MODEL_REPO,
        "--hf-token", HF_TOKEN,
    ]
    train_transformer()
    print("Transformer training complete!")

## Push training_complete marker

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj=json.dumps({"status": "complete", "vqvae": TRAIN_VQVAE, "transformer": TRAIN_TRANSFORMER}).encode(),
    path_in_repo="training_complete.json",
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print("Training marker pushed to HF Hub")